# Pattern 6: Swarm Pattern

> **Google Cloud Reference**: [Swarm Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#swarm-pattern)

Multiple specialized agents work **collaboratively** in an **all-to-all communication** structure, iteratively refining a solution.

**Key characteristics:**
- No single coordinator controls the workflow
- Any agent can hand off to any other agent
- Agents debate, critique, and build on each other's work
- A **dispatcher** routes the initial request (but doesn't orchestrate)
- Must have an explicit exit condition (max iterations, consensus, time limit)

```
User Request
     │
     ▼
[Dispatcher] ──► [Agent A] ◄──► [Agent B]
                    ▲  ▼           ▲  ▼
                    [Agent C] ◄──► [Agent D]
                         │
                         ▼
                  (consensus reached)
                         │
                         ▼
                    Final Answer
```

**When to use:** Ambiguous or complex problems benefiting from debate and multiple perspectives — design, strategy, creative work.

**Trade-offs:**
- ✅ Highest quality creative/strategic outputs
- ✅ Simulates a real expert team debate
- ⚠️ Most expensive and complex pattern
- ⚠️ Risk of infinite loops without good exit conditions
- ⚠️ Hard to predict or control execution flow

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### Example A: Product Design Swarm
Four specialist agents collaboratively design a new product by debating trade-offs.
Each agent can hand off to others and build on their contributions.

In [2]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat

model = OpenAIChat(id="google/gemini-2.5-flash")

# Each swarm agent has a distinct perspective and can reference/critique others

market_researcher = Agent(
    name="Market Researcher",
    role="Evaluate market opportunity, customer needs, and competitive positioning",
    model=model,
    instructions=[
        "Analyze market size, target segments, and unmet needs.",
        "Challenge ideas that don't address real customer pain points.",
        "Reference what competitors are doing and where the gaps are.",
        "When you see a technical proposal, ask: 'But who is the customer and what's the pain?'",
    ],
    markdown=True,
)

product_engineer = Agent(
    name="Product Engineer",
    role="Evaluate technical feasibility, architecture, and development complexity",
    model=model,
    instructions=[
        "Assess technical complexity: Easy / Medium / Hard / Very Hard.",
        "Propose a minimal viable architecture.",
        "Challenge over-engineered solutions — advocate for simplicity.",
        "When reviewing market analysis, identify which features are technically feasible in 3 months.",
    ],
    markdown=True,
)

financial_modeler = Agent(
    name="Financial Modeler",
    role="Model unit economics, pricing strategy, and financial viability",
    model=model,
    instructions=[
        "Build rough unit economics: CAC, LTV, payback period.",
        "Propose pricing tiers (freemium / pro / enterprise).",
        "Challenge any proposal without a clear path to profitability.",
        "Identify if the market size justifies the investment.",
    ],
    markdown=True,
)

ux_designer = Agent(
    name="UX Designer",
    role="Define user experience, onboarding flow, and product usability",
    model=model,
    instructions=[
        "Define the core user journey in 5 steps or fewer.",
        "Identify the 'aha moment' — when users first get value.",
        "Challenge features that add complexity without clear user benefit.",
        "Propose a 60-second onboarding flow.",
    ],
    markdown=True,
)

# Swarm — agents can freely interact, no central orchestrator
product_swarm = Team(
    name="Product Design Swarm",
    mode="collaborate",  # All-to-all communication — swarm pattern
    model=model,
    members=[market_researcher, product_engineer, financial_modeler, ux_designer],
    instructions=[
        "This is a collaborative product design session.",
        "Each expert should contribute from their domain AND actively challenge other experts.",
        "The goal: reach consensus on a product design specification.",
        "After each round of feedback, integrate the critiques and update the proposal.",
        "Produce a final unified Product Design Specification with: Problem, Solution, Target User, Tech Stack, Pricing, UX Flow, Go-to-Market.",
        "Maximum 3 rounds of debate — then synthesize a final spec.",
    ],
    
    markdown=True,
)

product_swarm.print_response(
    """
    Design a new B2B SaaS product that uses agentic AI to automate 
    legal contract review for small law firms (5-20 attorneys).
    
    Each expert: contribute your analysis AND actively debate the others.
    Conclude with a unified Product Design Specification.
    """,
    stream=True,
)

Output()

ERROR    Error from OpenAI API: JSON error injected into SSE stream

ERROR    Error in Agent run: JSON error injected into SSE stream

ERROR    Error from OpenAI API: JSON error injected into SSE stream

ERROR    Error in Agent run: JSON error injected into SSE stream

### Example B: Debate-Style Swarm
Agents explicitly debate a controversial technical decision, then reach consensus.

In [3]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat

model = OpenAIChat(id="google/gemini-2.5-flash")

# Debate: Should we use a microservices or monolith architecture?

microservices_advocate = Agent(
    name="Microservices Advocate",
    role="Champion microservices architecture",
    model=model,
    instructions=[
        "Argue for microservices: scalability, team autonomy, fault isolation.",
        "Actively counter the monolith advocate's points with specific rebuttals.",
        "Concede valid points from the opposition — intellectual honesty matters.",
    ],
    markdown=True,
)

monolith_advocate = Agent(
    name="Monolith Advocate",
    role="Champion monolithic architecture",
    model=model,
    instructions=[
        "Argue for monolith: simplicity, faster development, lower ops cost.",
        "Counter microservices points with concrete trade-off data.",
        "Concede valid points from the opposition — intellectual honesty matters.",
    ],
    markdown=True,
)

pragmatist = Agent(
    name="Pragmatist Mediator",
    role="Find the practical middle ground and synthesize the best decision",
    model=model,
    instructions=[
        "Listen to both sides and identify where they agree.",
        "Propose a 'modular monolith' or phased migration strategy if appropriate.",
        "Make the final decision recommendation based on the team's stated context.",
        "Be definitive — the team needs a clear answer, not more debate.",
    ],
    markdown=True,
)

architecture_debate = Team(
    name="Architecture Decision Swarm",
    mode="collaborate",
    model=model,
    members=[microservices_advocate, monolith_advocate, pragmatist],
    instructions=[
        "Conduct a structured architectural debate — 2 rounds max.",
        "Round 1: Each advocate presents their position.",
        "Round 2: Each advocate rebuts the other's points.",
        "Final: Pragmatist synthesizes and makes a final recommendation.",
    ],
    markdown=True,
)

architecture_debate.print_response(
    """
    Context: A 5-person startup building an AI-powered HR platform.
    They have 18 months runway and need to ship an MVP in 3 months.
    The team is 3 backend engineers and 2 frontend engineers.
    Expected users at launch: 50 companies, ~5000 employees.
    
    Question: Should they start with microservices or a monolith?
    Debate this and reach a consensus decision.
    """,
    stream=True,
)

Output()

ERROR    Error from OpenAI API: JSON error injected into SSE stream

ERROR    Error in Agent run: JSON error injected into SSE stream

ERROR    Error from OpenAI API: JSON error injected into SSE stream

ERROR    Error in Team run: JSON error injected into SSE stream

### Swarm vs Coordinator vs Hierarchical

| Dimension | Swarm | Coordinator | Hierarchical |
|-----------|-------|------------|-------------|
| Communication | All-to-all | Star (hub-spoke) | Tree (parent-child) |
| Central controller | None (dispatcher only) | Coordinator agent | Root agent |
| Decision making | Collective consensus | Coordinator decides | Root decomposes |
| Best for | Debate, creative, strategy | Routing diverse requests | Complex ambiguous tasks |
| Cost | Highest | Medium | High |
| Predictability | Lowest | High | Medium |